<a href="https://colab.research.google.com/github/M7office/Stroke/blob/main/AHA_similar_sis3_different_profiles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:

# ============================================================
# Slide 6 / Figure 6 — Similar SIS3 values can arise from distinct profiles
# Colab-ready script (v3)
#
# Inputs:
#   AHA_slide04_outputs_v8/slide04_patient_level_module_values.csv
#
# Outputs:
#   AHA_slide06_outputs_v3/
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle, Patch

BASE = Path.cwd()
OUT = BASE / "AHA_slide06_outputs_v8"
OUT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Styling
# -----------------------------
COLORS = {
    "black": "#222222",
    "gray": "#666666",
    "lightgray": "#D8D5CF",
    "verylight": "#F3F1ED",
    "row_alt": "#F7F5F1",
    "spine": "#444444",
    # subgroup colors
    "g1": "#4E79A7",   # blue
    "g2": "#F28E2B",   # orange
    "g3": "#59A14F",   # green
    "g4": "#B07AA1",   # purple
    "violin_fill": "#B8CCE0",
    "threshold": "#2E86DE",
    "meanline": "#C0392B",
    "errorbar": "#D98880",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.family": "DejaVu Sans",
    "font.size": 9.2,
    "axes.titlesize": 9.8,
    "axes.titleweight": "bold",
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.3,
    "legend.fontsize": 8.4,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 450,
})

SIS3_CUTOFF = 63
TIME_ORDER = ["0–1y", "1–3y", ">3y"]
PATHWAYS = [
    "Serotonin / tryptophan-kynurenine / monoamine metabolism",
    "Vesicle secretion / extracellular vesicle / membrane trafficking",
    "Cell death / cellular stress / proteostasis",
    "Metabolic / lipid / atherosclerosis / mitochondrial-energy biology",
    "mTOR / MAPK / NF-kB / growth-survival signaling",
    "Hormone / neuroendocrine / HPA-like systemic signaling",
    "Systemic organ injury / leakage / comorbidity markers",
    "Peripheral immune / inflammatory activation",
    "Complement / coagulation / platelet axis",
    "Endothelial / BBB / neurovascular unit",
    "Synaptic / neuronal plasticity / neurotrophic signaling",
    "Integrin / ECM / cell adhesion / vascular remodeling",
]
PATHWAY_LABELS = [f"{i+1:02d}. {p}" for i, p in enumerate(PATHWAYS)]
XBAR_LIM = 0.6

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(BASE / "AHA_slide04_outputs_v8" / "slide04_patient_level_module_values.csv")
df["time_bin"] = pd.Categorical(df["time_bin"], categories=TIME_ORDER, ordered=True)

low = df[df["sis3_group"] == "Lower SIS3 (≤63)"].copy()
g1 = low[low["time_bin"] == "0–1y"].copy().sort_values("sis3", ascending=False)
g2 = low[low["time_bin"] == "1–3y"].copy().sort_values("sis3", ascending=False)
g3_source = low[low["time_bin"] == ">3y"].copy().sort_values("sis3", ascending=False)
g3 = g3_source.head(4).copy()
g4 = g3_source.tail(3).copy()

subgroups = [
    ("0–1y", g1, COLORS["g1"]),
    ("1–3y", g2, COLORS["g2"]),
    (">3y top 4 SIS3", g3, COLORS["g3"]),
    (">3y bottom 3 SIS3", g4, COLORS["g4"]),
]

# Save subgroup membership
subgroup_rows = []
for short, sub, color in subgroups:
    for _, r in sub.iterrows():
        subgroup_rows.append({
            "subgroup": short,
            "patient_id": r["_patient_id_norm"],
            "sis3": r["sis3"],
            "time_years": r["time_years"],
            "time_bin": r["time_bin"],
            "color": color,
        })
pd.DataFrame(subgroup_rows).to_csv(OUT / "slide06_low_sis3_subgroup_membership.csv", index=False)

# Summary table
summary_rows = []
for short, sub, color in subgroups:
    row = {"subgroup": short, "n": len(sub), "color": color}
    for p in PATHWAYS:
        row[p] = float(pd.to_numeric(sub[p], errors="coerce").mean()) if len(sub) else np.nan
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "slide06_profile_summary.csv", index=False)

# -----------------------------
# Statistical helper
# -----------------------------
def bootstrap_mean_diff_p(x, y, n_boot=4000, seed=7):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna().to_numpy(dtype=float)
    y = pd.to_numeric(pd.Series(y), errors="coerce").dropna().to_numpy(dtype=float)
    if len(x) < 2 or len(y) < 2:
        return np.nan, np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    obs = float(np.mean(x) - np.mean(y))
    xb = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    yb = rng.choice(y, size=(n_boot, len(y)), replace=True).mean(axis=1)
    diffs = xb - yb
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return obs, float(lo), float(hi), float(p)

comparisons = [
    ("1–3y − 0–1y", g2, g1),
    (">3y top4 − 0–1y", g3, g1),
    (">3y bottom3 − 0–1y", g4, g1),
    (">3y top4 − 1–3y", g3, g2),
    (">3y bottom3 − 1–3y", g4, g2),
    (">3y bottom3 − top4", g4, g3),
]

stat_rows = []
for idx, pth in enumerate(PATHWAYS):
    row = {"No.": f"{idx+1:02d}", "Biological module": pth}
    for cname, A, B in comparisons:
        diff, lo, hi, p = bootstrap_mean_diff_p(A[pth], B[pth], seed=100 + idx)
        row[f"{cname} p"] = p
        row[f"{cname} diff"] = diff
        row[f"{cname} lo"] = lo
        row[f"{cname} hi"] = hi
    stat_rows.append(row)
stats_df = pd.DataFrame(stat_rows)
stats_df.to_csv(OUT / "slide06_pairwise_pvalues.csv", index=False)

# -----------------------------
# Figure 6A: bar panels only
# -----------------------------
fig = plt.figure(figsize=(13.8, 6.8))
gs = GridSpec(
    3, 4,
    figure=fig,
    height_ratios=[0.45, 4.35, 1.15],
    width_ratios=[1.58, 1.22, 1.22, 1.22],
    hspace=0.08,
    wspace=0.24,
)

bar_axes = [fig.add_subplot(gs[1, i]) for i in range(4)]

# shared y setup
y = np.arange(len(PATHWAYS))

for idx, (ax, (short, sub, color)) in enumerate(zip(bar_axes, subgroups)):
    means = [summary.loc[summary["subgroup"] == short, p].iloc[0] for p in PATHWAYS]
    ax.barh(y, means, color=color, edgecolor=color, linewidth=0.6)
    ax.axvline(0, color=COLORS["spine"], lw=0.8)
    ax.set_xlim(-XBAR_LIM, XBAR_LIM)
    ax.set_xticks(np.arange(-0.4, 0.41, 0.2))
    ax.set_axisbelow(True)
    ax.grid(axis="x", color=COLORS["verylight"], lw=0.6, zorder=0)
    ax.set_ylim(-0.5, len(PATHWAYS)-0.5)
    ax.invert_yaxis()
    ax.set_title(f"{short}\n(n={len(sub)})", pad=8)
    ax.set_xlabel("Mean standardized\nbiological-module value", labelpad=6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if idx == 0:
        ax.set_yticks(y)
        ax.set_yticklabels(PATHWAY_LABELS)
        ax.set_ylabel("Biological module")
        ax.tick_params(axis="y", labelsize=8.2)
    else:
        ax.set_yticks(y)
        ax.set_yticklabels([])
        ax.tick_params(axis="y", left=True, length=3, color=COLORS["gray"])

fig.suptitle("Figure 6A. Similar SIS3 Values Can Arise From Distinct Biological-Module Profiles",
             x=0.055, y=0.975, ha="left", fontsize=13.0, fontweight="bold", color=COLORS["black"])

captionA = (
    "Bars show mean standardized biological-module values for 4 lower-SIS3 subgroups: 0–1 year (n=6), "
    "1–3 years (n=4), >3 years with the top 4 SIS3 values (n=4), and >3 years with the bottom 3 SIS3 values (n=3)."
)
fig.text(0.055, 0.03, captionA, ha="left", va="bottom", fontsize=8.0, color=COLORS["gray"], wrap=True)

plt.subplots_adjust(left=0.23, right=0.99, top=0.90, bottom=0.14)
baseA = "figure6A_distinct_profiles_barplot_v8"
for ext in ["png", "pdf", "svg"]:
    fig.savefig(OUT / f"{baseA}.{ext}", bbox_inches="tight")
plt.close(fig)

# -----------------------------
# Table 6: P-value table
# forest-table style, but no forest column and no FDR
# -----------------------------
fig = plt.figure(figsize=(16.8, 6.9))
gs = GridSpec(
    3, 8,
    figure=fig,
    height_ratios=[0.42, 4.30, 1.35],
    width_ratios=[0.55, 3.40, 0.92, 1.02, 1.08, 1.08, 1.12, 1.08],
    hspace=0.04,
    wspace=0.05,
)

ax_no = fig.add_subplot(gs[1, 0])
ax_mod = fig.add_subplot(gs[1, 1], sharey=ax_no)
axes_p = [fig.add_subplot(gs[1, i], sharey=ax_no) for i in range(2, 8)]

# alternating row shading
y = np.arange(len(PATHWAYS))
for ax in [ax_no, ax_mod] + axes_p:
    ax.set_ylim(-0.7, len(PATHWAYS)-0.3)
    ax.invert_yaxis()
    for yi in y:
        if yi % 2 == 1:
            ax.axhspan(yi - 0.5, yi + 0.5, color=COLORS["verylight"], zorder=0)

# No. column
ax_no.set_xlim(0, 1)
ax_no.axis("off")
ax_no.text(0.00, 1.02, "No.", transform=ax_no.transAxes,
           ha="left", va="bottom", fontsize=9.0, fontweight="bold", color=COLORS["black"])
for yi, no in zip(y, stats_df["No."]):
    ax_no.text(0.00, yi, no, ha="left", va="center", fontsize=8.5, color=COLORS["black"])

# Module column
ax_mod.set_xlim(0, 1)
ax_mod.axis("off")
ax_mod.text(0.00, 1.02, "Biological module", transform=ax_mod.transAxes,
            ha="left", va="bottom", fontsize=9.0, fontweight="bold", color=COLORS["black"])
for yi, mod in zip(y, stats_df["Biological module"]):
    ax_mod.text(0.00, yi, mod, ha="left", va="center", fontsize=8.4, color=COLORS["black"])

# p columns
p_colnames = [
    "1–3y − 0–1y",
    ">3y top4 − 0–1y",
    ">3y bottom3 − 0–1y",
    ">3y top4 − 1–3y",
    ">3y bottom3 − 1–3y",
    ">3y bottom3 − top4",
]
for ax, cname in zip(axes_p, p_colnames):
    ax.set_xlim(0, 1)
    ax.axis("off")
    header = cname.replace(" − ", " −\n")
    ax.text(0.00, 1.02, header, transform=ax.transAxes,
            ha="left", va="bottom", fontsize=8.2, fontweight="bold", color=COLORS["black"])
    vals = stats_df[f"{cname} p"].tolist()
    for yi, p in zip(y, vals):
        txt = "NA" if pd.isna(p) else (f"{p:.3f}" if p >= 0.001 else "<0.001")
        ax.text(0.00, yi, txt, ha="left", va="center", fontsize=8.3, color=COLORS["black"])

fig.suptitle("Table 6. Pairwise P Values for Biological-Module Comparisons Across Lower-SIS3 Subgroups",
             x=0.055, y=0.975, ha="left", fontsize=13.0, fontweight="bold", color=COLORS["black"])

captionT = (
    "P values were estimated by bootstrap resampling of the mean difference in standardized biological-module values "
    "for each pairwise subgroup comparison. No forest column and no FDR correction are shown in this summary table."
)
fig.text(0.055, 0.03, captionT, ha="left", va="bottom", fontsize=8.0, color=COLORS["gray"], wrap=True)

plt.subplots_adjust(left=0.06, right=0.99, top=0.90, bottom=0.14)
baseT = "table6_pairwise_pvalues_subgroups_v8"
for ext in ["png", "pdf", "svg"]:
    fig.savefig(OUT / f"{baseT}.{ext}", bbox_inches="tight")
plt.close(fig)

# -----------------------------
# Figure 6B (violin only)
# -----------------------------
fig, ax = plt.subplots(figsize=(7.2, 5.8))

time_groups = [low.loc[low["time_bin"] == tb, "sis3"].dropna().values for tb in TIME_ORDER]
positions = np.arange(1, 4)

vp = ax.violinplot(
    time_groups, positions=positions, widths=0.75,
    showmeans=False, showmedians=False, showextrema=False
)
for body in vp["bodies"]:
    body.set_facecolor(COLORS["violin_fill"])
    body.set_edgecolor("none")
    body.set_alpha(0.85)

# Means / IQR
for pos, arr in zip(positions, time_groups):
    arr = np.asarray(arr, dtype=float)
    mean = np.mean(arr)
    q1, q3 = np.percentile(arr, [25, 75])
    ax.vlines(pos, q1, q3, color=COLORS["errorbar"], lw=1.6, zorder=3)
    ax.hlines(mean, pos - 0.16, pos + 0.16, color=COLORS["meanline"], lw=1.6, zorder=4)
    # n labels above the threshold line
    ax.text(pos, SIS3_CUTOFF + 2.0, f"n={len(arr)}", ha="center", va="bottom",
            fontsize=8.8, color=COLORS["black"])

rng = np.random.default_rng(7)
def jitter(n, width=0.09):
    return rng.uniform(-width, width, size=n)

# Scatter by subgroup colors
for pos, sub, color in [(1, g1, COLORS["g1"]), (2, g2, COLORS["g2"])]:
    x = np.full(len(sub), pos) + jitter(len(sub))
    ax.scatter(x, sub["sis3"], s=24, color=color, edgecolor="none", zorder=5)

x = np.full(len(g3), 3.0) + jitter(len(g3))
ax.scatter(x, g3["sis3"], s=26, color=COLORS["g3"], edgecolor="none", zorder=5)
x = np.full(len(g4), 3.0) + jitter(len(g4))
ax.scatter(x, g4["sis3"], s=26, color=COLORS["g4"], edgecolor="none", zorder=5)

ax.axhline(SIS3_CUTOFF, color=COLORS["threshold"], lw=1.0, ls="--")
ax.text(3.18, SIS3_CUTOFF - 1.2, "SIS3 = 63", color=COLORS["spine"], fontsize=8.5, va="top")

ax.set_xlim(0.45, 3.55)
ax.set_ylim(0, 105)
ax.set_xticks(positions)
ax.set_xticklabels(TIME_ORDER)
ax.set_xlabel("Time since stroke")
ax.set_ylabel("SIS3")
ax.set_title("Figure 6B. Similar SIS3 Values Can Arise From Distinct Profile Configurations",
             loc="left", pad=10, fontsize=12.0, fontweight="bold")
ax.set_axisbelow(True)
ax.grid(axis="y", color=COLORS["verylight"], lw=0.6, zorder=0)

legend_handles = [
    Patch(facecolor=COLORS["g1"], edgecolor="none", label="0–1y"),
    Patch(facecolor=COLORS["g2"], edgecolor="none", label="1–3y"),
    Patch(facecolor=COLORS["g3"], edgecolor="none", label=">3y top 4 SIS3"),
    Patch(facecolor=COLORS["g4"], edgecolor="none", label=">3y bottom 3 SIS3"),
]
# Put legend beneath plot but clearly above caption
ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.11),
          ncol=2, frameon=False, columnspacing=1.6, handlelength=1.2)

caption_b = (
    "Violin plots summarize the SIS3 distribution among lower-SIS3 patients across time bins. "
    "Scatter points are colored to match the 4 subgroup definitions used in Figure 6A."
)
fig.text(0.11, 0.01, caption_b, ha="left", va="bottom", fontsize=8.0, color=COLORS["gray"], wrap=True)
plt.subplots_adjust(left=0.11, right=0.98, top=0.89, bottom=0.27)

baseB = "figure6B_low_sis3_violin_colored_subgroups_v8"
for ext in ["png", "pdf", "svg"]:
    fig.savefig(OUT / f"{baseB}.{ext}", bbox_inches="tight")
plt.close(fig)

print("Created Slide 6 outputs:")
for p in sorted(OUT.glob("figure6*.*")):
    print(" -", p.name)
for p in sorted(OUT.glob("table6*.*")):
    print(" -", p.name)
for p in sorted(OUT.glob("slide06*.*")):
    print(" -", p.name)


Created Slide 6 outputs:
 - figure6A_distinct_profiles_barplot_v8.pdf
 - figure6A_distinct_profiles_barplot_v8.png
 - figure6A_distinct_profiles_barplot_v8.svg
 - figure6B_low_sis3_violin_colored_subgroups_v8.pdf
 - figure6B_low_sis3_violin_colored_subgroups_v8.png
 - figure6B_low_sis3_violin_colored_subgroups_v8.svg
 - table6_pairwise_pvalues_subgroups_v8.pdf
 - table6_pairwise_pvalues_subgroups_v8.png
 - table6_pairwise_pvalues_subgroups_v8.svg
 - slide06_low_sis3_subgroup_membership.csv
 - slide06_pairwise_pvalues.csv
 - slide06_profile_summary.csv
